# 35 RL efficiency phase: evaluation (CPU runtime)

Scores the efficiency-phase models (after 5, 10 and 15 iterations) on validation, dev and mini-dev, compares them on validation with the RL model they started from (same rule as before), and prints the training log and the full report. For this phase, look especially at the efficiency columns: geometric mean cost, answers at least 2x cheaper than gold, and how often a model's query is cheaper or costlier than the starting model's on the questions both answer correctly.

**Runtime > Change runtime type > CPU**.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# Starts from the RL model kept on validation (33_rl_evaluate).
INIT = json.load(open(f"{R}/rl_v1_choice/selection.json"))["winner"]
RUN = "rl_eff_v1"
# Snapshots after 5 and 10 of the 15 iterations (own runs), then the final model.
RUNS = [r for r in (f"{RUN}_it5", f"{RUN}_it10") if os.path.exists(f"{R}/{r}/adapter_meta.json")] + [RUN]
print("starts from:", INIT, "| run:", RUN)
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
METHOD = "rl-eff-" + json.load(open(f"{R}/{RUN}/adapter_meta.json"))["arm"]
meta = json.load(open(f"{R}/{EXPORT}/export_meta.json"))
VAL_DBS = ",".join(meta["holdout_db_ids"])

### 1. Stage the validation databases and the dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 2. Score the models (resumable)

In [ ]:
for r in RUNS:
    !python -m effsql.train eval-val --run {r}
    !effsql evaluate --pred {R}/{r}/predictions_dev.jsonl --split dev --timing none --run {r}_dev --model {MODEL} --method {METHOD}
    !effsql evaluate --pred {R}/{r}/predictions_minidev.jsonl --split minidev --timing quick --run {r}_minidev --model {MODEL} --method {METHOD}

### 3. Against the starting RL model on validation

In [ ]:
!python -m effsql.train select --runs {','.join([INIT] + RUNS)} --out rl_eff_v1_choice

### 4. Training log: per iteration, on the answers the model wrote while training

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # installed after this Python session started: not on its path yet
from effsql.train.rl import format_summary
print(format_summary(json.load(open(f"{R}/{RUN}/rl_summary.json"))))

### 5. All results, each model against the starting RL model

Main metric: **R-VES** (BIRD's reward per question with step counts in place of seconds: at least 2x faster than gold 112, 1 to 2x 100, up to 2x slower 87, 2 to 4x slower 71, slower 50, wrong 0; gold = 100), with a 95% bootstrap interval. **R-VES/EX**: the same over correct answers only, the efficiency apart from accuracy. **VES** (100 x square root of the speed-up, unbounded) is shown but a few extreme questions drive it. Then each model against the reference on the same questions: R-VES difference (Wilcoxon test), EX (McNemar test), and how often its query is cheaper or costlier by at least 10% when both are correct (sign test). Nothing is executed: the table reads the saved evaluations. Saved as `report.txt` and `report.json`.

In [ ]:
REPORT = [INIT, "pool_v1_{split}_base", f"{EXPORT}_best_e1", "rl_v1"] + RUNS
LABELS = ",".join(["pool_v1_{split}_base=zero-shot"] + [f"{r}={r.replace(EXPORT + '_', '')}" for r in REPORT if "{split}" not in r])
REF, OUT = INIT, "rl_eff_v1_report"
!python -m effsql.train report --runs "{','.join(REPORT)}" --ref {REF} --labels "{LABELS}" --out {OUT}

### Done: disconnect the runtime